# 04-3 실습 — 텍스트 파일·모드·with·줄바꿈

작업 목록을 UTF-8 텍스트로 저장하고 한 줄씩 다시 읽으며, 실행 이력을 별도 파일에 추가합니다. 모든 입출력은 새 임시 디렉터리 안에서만 수행합니다.

## Goal

- r, w, a, x 모드의 생성·덮어쓰기·추가 정책을 구분합니다.
- with가 파일을 닫지만 이미 쓴 내용을 되돌리지는 않음을 확인합니다.
- 작은 파일의 전체 읽기와 큰 파일의 줄 단위 반복을 구분합니다.
- write, writelines, join의 줄바꿈 책임을 확인합니다.
- 의미 있는 공백과 원본 줄바꿈을 보존하는 방법을 연습합니다.

## Setup

TemporaryDirectory와 Python 표준 라이브러리만 사용합니다. 노트북을 처음부터 다시 실행하면 x와 a 모드 실습도 깨끗한 파일에서 시작합니다.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

_lab_context = TemporaryDirectory(prefix='chapter04-text-')
LAB_ROOT = Path(_lab_context.name).resolve(strict=True)
TEXT_DIR = LAB_ROOT / 'text'
TEXT_DIR.mkdir()

def expect_raises(expected_exception, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except expected_exception as exc:
        return exc
    except Exception as exc:
        raise AssertionError(
            f'{expected_exception.__name__} 대신 {type(exc).__name__} 발생'
        ) from exc
    raise AssertionError(f'{expected_exception.__name__}이 발생하지 않았습니다.')

print('격리된 실습 루트:', LAB_ROOT)

## Steps

### 1. 파일 모드와 with의 책임 확인

| 모드 | 파일이 없을 때 | 파일이 있을 때 |
| --- | --- | --- |
| r | FileNotFoundError | 읽음 |
| w | 새로 만듦 | 여는 시점에 기존 내용을 비움 |
| a | 새로 만듦 | 끝에 추가 |
| x | 새로 만듦 | FileExistsError |

with는 정상·예외 경로에서 파일을 닫지만 논리적 롤백은 제공하지 않습니다.

In [ ]:
resource_path = TEXT_DIR / 'resource.txt'
with resource_path.open('w', encoding='utf-8') as file:
    assert not file.closed
    file.write('정상 종료\n')
assert file.closed

truncated_path = TEXT_DIR / 'truncated.txt'
truncated_path.write_text('기존 내용\n', encoding='utf-8')
with truncated_path.open('w', encoding='utf-8'):
    pass
assert truncated_path.read_text(encoding='utf-8') == ''

partial_path = TEXT_DIR / 'partial.txt'
partial_path.write_text('안정된 내용\n', encoding='utf-8')
try:
    with partial_path.open('w', encoding='utf-8') as file:
        file.write('일부만 기록')
        raise RuntimeError('의도한 실패')
except RuntimeError:
    pass

assert file.closed
assert partial_path.read_text(encoding='utf-8') == '일부만 기록'
print('with는 파일을 닫았지만 기록을 되돌리지 않았습니다.')

### 2. 작업 목록과 실행 이력 함수

파일을 열기 전에 모든 항목을 먼저 검증해야 중간까지 쓴 파일이 남지 않습니다. 작업 목록은 x 모드로 새로 만들고, 읽을 때는 파일 객체를 직접 반복합니다.

In [ ]:
def require_single_line(value, label):
    if not isinstance(value, str):
        raise TypeError(f'{label}은 문자열이어야 합니다.')
    if '\n' in value or '\r' in value:
        raise ValueError(f'{label}에는 줄바꿈을 포함할 수 없습니다.')
    return value

def save_new_tasks(path, tasks):
    checked_tasks = [
        require_single_line(task, '작업 항목')
        for task in tasks
    ]
    with Path(path).open('x', encoding='utf-8', newline='\n') as file:
        for task in checked_tasks:
            file.write(task + '\n')

def load_numbered_tasks(path):
    records = []
    with Path(path).open('r', encoding='utf-8', newline=None) as file:
        for line_number, line in enumerate(file, start=1):
            records.append((line_number, line.removesuffix('\n')))
    return records

def append_history(path, message):
    checked_message = require_single_line(message, '이력 메시지')
    with Path(path).open('a', encoding='utf-8', newline='\n') as file:
        file.write(checked_message + '\n')

### 3. 입력을 바꾸며 완성 예제 실행

tasks의 문자열을 다른 한 줄짜리 작업으로 바꾸거나 항목을 추가한 뒤 다시 처음부터 실행해 보세요. 두 번째 항목의 선행 공백은 의미 있는 데이터로 보존됩니다.

In [ ]:
tasks = ['로그 확인', '  공백 보존', '보고서 저장']
tasks_path = TEXT_DIR / 'tasks.txt'
history_path = TEXT_DIR / 'history.log'

save_new_tasks(tasks_path, tasks)
numbered_tasks = load_numbered_tasks(tasks_path)

assert numbered_tasks == list(enumerate(tasks, start=1))
assert numbered_tasks[1][1].startswith('  ')
expect_raises(FileExistsError, save_new_tasks, tasks_path, tasks)

history_messages = ['작업 목록 읽기 성공', '검증 완료']
for message in history_messages:
    append_history(history_path, message)

assert history_path.read_text(encoding='utf-8').splitlines() == history_messages
print(numbered_tasks)
print(history_path.read_text(encoding='utf-8'), end='')

### 4. write·writelines·join의 줄바꿈 책임

writelines는 이름과 달리 줄바꿈을 자동으로 넣지 않습니다. 줄바꿈 없는 값의 목록은 join으로 출력 계약을 눈에 보이게 만듭니다.

In [ ]:
joined_path = TEXT_DIR / 'joined.txt'
with joined_path.open('w', encoding='utf-8', newline='\n') as file:
    file.writelines(['alpha', 'beta'])
assert joined_path.read_text(encoding='utf-8') == 'alphabeta'

items = ['alpha', 'beta', 'gamma']
with joined_path.open('w', encoding='utf-8', newline='\n') as file:
    file.write('\n'.join(items) + '\n')
assert joined_path.read_text(encoding='utf-8') == 'alpha\nbeta\ngamma\n'

append_boundary_path = TEXT_DIR / 'append-boundary.log'
append_boundary_path.write_text('시작', encoding='utf-8')
append_history(append_boundary_path, '다음')
assert append_boundary_path.read_text(encoding='utf-8') == '시작다음\n'
print('기존 파일의 마지막 줄 경계를 모르면 레코드가 이어질 수 있습니다.')

### 5. 줄바꿈 정규화와 원본 보존

newline=None은 여러 줄 끝을 줄바꿈 문자 하나로 정규화합니다. newline=''은 파일에 있던 LF, CRLF, CR을 그대로 반환합니다.

In [ ]:
mixed_path = TEXT_DIR / 'mixed-lines.txt'
with mixed_path.open('w', encoding='utf-8', newline='') as file:
    file.write('alpha\nbeta\r\ngamma\r')

with mixed_path.open('r', encoding='utf-8', newline=None) as file:
    normalized = list(file)

with mixed_path.open('r', encoding='utf-8', newline='') as file:
    original_endings = list(file)

assert normalized == ['alpha\n', 'beta\n', 'gamma\n']
assert original_endings == ['alpha\n', 'beta\r\n', 'gamma\r']
print('정규화:', [repr(line) for line in normalized])
print('원본 보존:', [repr(line) for line in original_endings])

## Checks

빈 목록, 줄바꿈이 포함된 항목, 없는 파일, 마지막 줄에 줄바꿈이 없는 파일을 점검합니다. PermissionError는 운영체제와 실행 계정에 따라 달라지므로 여기서는 강제로 재현하지 않습니다.

In [ ]:
empty_tasks_path = TEXT_DIR / 'empty-tasks.txt'
save_new_tasks(empty_tasks_path, [])
assert empty_tasks_path.read_bytes() == b''
assert load_numbered_tasks(empty_tasks_path) == []

expect_raises(
    ValueError,
    save_new_tasks,
    TEXT_DIR / 'bad-newline.txt',
    ['정상', '두 줄\n항목'],
)
expect_raises(
    ValueError,
    append_history,
    TEXT_DIR / 'bad-history.log',
    '잘못된\r메시지',
)
expect_raises(FileNotFoundError, load_numbered_tasks, TEXT_DIR / 'missing.txt')
assert not (TEXT_DIR / 'bad-newline.txt').exists()
assert not (TEXT_DIR / 'bad-history.log').exists()

no_final_newline = TEXT_DIR / 'no-final-newline.txt'
with no_final_newline.open('w', encoding='utf-8', newline='\n') as file:
    file.write('첫 줄\n마지막 줄')
assert load_numbered_tasks(no_final_newline) == [
    (1, '첫 줄'),
    (2, '마지막 줄'),
]

sample = '  ALLOW  \n'
assert sample.removesuffix('\n') == '  ALLOW  '
assert sample.strip() == 'ALLOW'

assert tasks_path.read_text(encoding='utf-8').splitlines() == tasks
assert history_path.read_text(encoding='utf-8').splitlines() == history_messages
print('04-3 자기점검을 모두 통과했습니다.')

In [ ]:
_lab_context.cleanup()
assert not LAB_ROOT.exists()

## Next Steps

04-4에서는 같은 파일의 원본 바이트, 인코딩, 파일 위치와 고정 길이 구조를 다룹니다. 큰 파일은 read_text나 readlines로 모두 누적하지 말고 파일 객체를 직접 반복하며, 중요한 결과 파일의 완전한 저장은 04-8의 임시 파일 교체 절차를 사용합니다.